# Day 4: BERT and RoBERTa for Text Classification

Every from scratch model in this project hit the same wall. 222 training rows is too few, so everything overfits. Week 3's ANN got 39 to 52 percent, the LSTM got 50 percent.

Fine-tuning is different. DistilBERT already learned English from billions of words during its pretraining, so we're only nudging it to fit our 6 categories, not teaching it language from zero. This is the one setup where a small dataset actually works.

Using DistilBERT instead of full BERT since it's about half the size and runs about twice as fast, which matters on CPU.

In [1]:
import ssl
import certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification

torch.manual_seed(42)

df = pd.read_csv("news_dataset.csv")

X_train_text, X_test_text, y_train_labels, y_test_labels = train_test_split(
    df["Title"], df["Category"], test_size=0.2, random_state=42
)

label_encoder = LabelEncoder()
label_encoder.fit(df["Category"])
y_train = label_encoder.transform(y_train_labels)
y_test = label_encoder.transform(y_test_labels)
num_classes = len(label_encoder.classes_)

print("Train size:", len(X_train_text))
print("Test size:", len(X_test_text))
print("Classes:", list(label_encoder.classes_))

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print("Tokenizer loaded.")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train size: 222
Test size: 56
Classes: ['Business', 'Energy', 'Health', 'Markets', 'Politics', 'Technology']


Tokenizer loaded.


The SSL lines at the top are the same fix nltk and gensim needed earlier this week, the Python install was missing a valid cert bundle.

We use DistilBERT's own tokenizer, not our Day 1 preprocessing. It has its own vocabulary and its own way of splitting text that has to match how the model was trained.

In [2]:
##tokenize headlines using DistilBERT tokenizer
train_encodings = tokenizer(
    list(X_train_text),
    truncation=True,
    padding=True, ##pad shorter headlines with a special padding token so every row is the same lenght
    max_length=32, ##cap any headline longer than 32 tokens (this dataset has short headlines so will rarely trigger)
    return_tensors="pt", ##give back the PyTorch tensors directly instead of plain lists
)
test_encodings = tokenizer(
    list(X_test_text),
    truncation=True,
    padding=True,
    max_length=32,
    return_tensors="pt",
)
print("Input IDs shape:", train_encodings["input_ids"].shape)
print("Attention mask shape:", train_encodings["attention_mask"].shape)
print("\nFirst headline:", X_train_text.iloc[0])
print("Its input IDs:", train_encodings["input_ids"][0])
print("Its attention mask:", train_encodings["attention_mask"][0])

Input IDs shape: torch.Size([222, 32])
Attention mask shape: torch.Size([222, 32])

First headline: China's Xi to bring large CEO delegation on US visit, sources say
Its input IDs: tensor([  101,  2859,  1005,  1055,  8418,  2000,  3288,  2312,  5766, 10656,
         2006,  2149,  3942,  1010,  4216,  2360,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0])
Its attention mask: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])


The output shows the three concepts from today.

Input IDs are the token numbers. 101 is the [CLS] token at the start, 102 is [SEP] at the end, and the 0s are padding.

Subword tokenization is visible here. "China's" became three tokens, one for "china", one for the apostrophe, one for "s". The tokenizer splits complex or unfamiliar pieces into smaller known chunks. This is the same thing the Day 3 notes talked about.

The attention mask is 1 for real tokens and 0 for padding. That's how the model knows to ignore the padding during attention so no word wastes attention on filler.

In [3]:
##DistilBERT
##load pretrained DistilBERT + a fresh untrained classification head on top
##the body already knows English from pretraining, the head (a small linear layer -> 6 categories) starts random
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_classes,  ##tells it how many output categories the new head should have
)
##the "newly initialized weights" warning it prints is expected -- that's the random classification head
print(model.config.architectures)
print("Number of labels:", model.config.num_labels)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8895.09it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


['DistilBertForMaskedLM']
Number of labels: 6


The load report shows UNEXPECTED and MISSING keys, and it's a good picture of what fine-tuning is.

UNEXPECTED (vocab_transform, vocab_projector) is the masked language modeling head from the original pretraining, the part that guessed a hidden word from context. Our classification model has nowhere to put it so it gets thrown away.

MISSING (pre_classifier, classifier) is the classification head our model needs but the checkpoint doesn't have, since the pretrained model was never trained to classify anything. So it gets created fresh with random values.

The part with no warning is the actual DistilBERT body, all the attention layers. That loads perfectly and it's the part that already knows English. So the whole idea is keep the pretrained body, bolt on a random head, then train.

In [4]:
from torch.utils.data import Dataset, DataLoader

##wraps the tokenizer output + labels so PyTorch can loop through them in batches
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}  ##pulls input_ids + attention_mask for one headline
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = NewsDataset(train_encodings, y_train)
test_dataset = NewsDataset(test_encodings, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [5]:
##fine-tuning: small learning rate on purpose since we're nudging a pretrained model, not training from scratch
##first tried lr=2e-5, epochs=4 (the "standard" BERT fine-tuning recipe) -> only 57% accuracy, and the loss was
##still clearly dropping (1.65 -> 1.14), so the model was undertrained. that recipe assumes a bigger dataset
##with only 222 examples each epoch is tiny (14 batches), so more epochs are fine before it overfits.
##bumped to lr=3e-5, epochs=15 -> loss got down to 0.03 and accuracy jumped to 69.6%
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
epochs = 15

model.train()
for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(**batch)  ##HuggingFace models take input_ids, attention_mask, labels as keyword args and return loss directly
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss / len(train_loader):.4f}")

Epoch 1: loss=1.5978


Epoch 2: loss=1.3721


Epoch 3: loss=1.2137


Epoch 4: loss=0.9763


Epoch 5: loss=0.6986


...

Epoch 11: loss=0.0822


Epoch 12: loss=0.0626


Epoch 13: loss=0.0468


Epoch 14: loss=0.0373


Epoch 15: loss=0.0306


The first run used the standard BERT recipe, lr 2e-5 for 4 epochs. That only got 57 percent and the loss was still dropping, so it hadn't finished learning. That recipe is built for bigger datasets. With only 222 rows each epoch is just 14 batches, so it's cheap to run way more of them before overfitting kicks in.

Bumped it to lr 3e-5 for 15 epochs and the loss dropped from about 1.6 all the way to 0.03. The training loop itself is the same forward, loss, backward, step pattern as Week 3, the main difference is model(**batch) hands the whole tokenizer dict straight to the model and it returns the loss directly.

In [6]:
##evaluation portion
from sklearn.metrics import accuracy_score, classification_report

model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in test_loader:
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=1)  ##logits = raw class scores, argmax picks the winner
        all_preds.extend(preds.tolist())
        all_labels.extend(batch["labels"].tolist())

print("\nTest accuracy:", accuracy_score(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_, zero_division=0))


Test accuracy: 0.6964285714285714
              precision    recall  f1-score   support

    Business       0.67      0.82      0.74        17
      Energy       0.00      0.00      0.00         3
      Health       0.50      0.33      0.40         3
     Markets       0.78      0.90      0.84        20
    Politics       0.00      0.00      0.00         2
  Technology       0.60      0.55      0.57        11

    accuracy                           0.70        56
   macro avg       0.42      0.43      0.42        56
weighted avg       0.63      0.70      0.66        56



In [7]:
model.save_pretrained("distilbert_news_classifier")
tokenizer.save_pretrained("distilbert_news_classifier")
print("Saved fine-tuned model to distilbert_news_classifier/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 13.61it/s]

Saved fine-tuned model to distilbert_news_classifier/


## Takeaway

DistilBERT fine-tuned hit 69.6 percent. Here's how that stacks up against everything else.

| Approach | Accuracy |
| --- | --- |
| CountVectorizer (Day 1) | 71.4% |
| DistilBERT fine-tuned | 69.6% |
| TF-IDF (Day 1) | 62.5% |
| Plain averaged GloVe (Day 2) | 58.9% |
| Week 3 LSTM | 50% |
| Week 3 ANN (from scratch) | 39 to 52% |

This is the first deep learning model in the whole project that actually competes with the simple classical methods. It basically matches CountVectorizer and beats everything else deep learning. It's also the only model all project that got a Health prediction right in the test set, and Technology recall finally looks decent instead of zero.

The reason it works where Week 3 failed is the pretraining. Week 3's models had to learn English and the task at the same time from 222 rows, which is impossible. DistilBERT already knew English, so 222 rows was enough to just teach it which words point to which category. Fine-tuning a pretrained model is the right tool for a small dataset, training one from scratch is not.